In [1]:
import sys, os
import numpy as np
import pandas as pd
import geopandas as gpd
import datetime as dt
from survey import Survey, nine_to_county, purp_num_to_name18, purp_num_to_name23, mode_num_to_name23, county_order
from processed_survey import ProcessedSurvey
from survey_completeness import flag_complete, PERSON_REQS, HH_REQS, DAY_REQS, TRIP_REQS
sys.path.insert(0, r'Y:\champ\util\pythonlib-migration\master_versions\misc_utils')
from df_utils import df_to_excel

from openpyxl import load_workbook
from openpyxl.cell import MergedCell

In [2]:
SURVEY_2023 = {'household':{'filepath_or_buffer':r'..\..\..\Review_20241127\flagged\hh.csv'},
               'person':{'filepath_or_buffer':r'..\..\..\Review_20241127\flagged\person.csv'},
               'day':{'filepath_or_buffer':r'..\..\..\Review_20241127\flagged\day.csv'},
               'trip':{'filepath_or_buffer':r'..\..\..\Review_20241127\flagged\trip.csv'},
               'vehicle':{'filepath_or_buffer':r'..\..\..\Deliverable_20241127\vehicle.csv'},
               'location':{'filepath_or_buffer':r'..\..\..\Deliverable_20241127\location.csv'},
               }
OUTDIR = r'..\..\..\..\Review_20241127'
COUNTIES = r'Q:\GIS\Boundaries\Counties\Counties.shp'
FIPS_COUNTIES   =['001','013','041','055','075','081','085','095','097']

In [3]:
def write_to_excel(template_path, output_path, table_data):
    """
    Populate tables in an Excel template based on table titles in the Excel sheet.

    Parameters:
        template_path (str): Path to the Excel template.
        output_path (str): Path to save the updated Excel file.
        table_data (dict): Dictionary where keys are table titles in the Excel template
                           and values are Pandas DataFrames to populate the tables.
    """
    # Load the Excel template
    wb = load_workbook(template_path)
    ws = wb.active  # Assuming all tables are in the active sheet

    for table_title, df in table_data.items():
        # Locate the table title in the Excel template
        title_row, title_col = None, None
        for row in ws.iter_rows():
            for cell in row:
                if cell.value == table_title:  # Match the table title
                    title_row, title_col = cell.row, cell.column
                    break
            if title_row:
                break

        if not title_row or not title_col:
            print(f"Table title '{table_title}' not found in template!")
            continue

        # Write the DataFrame to the table below the title
        start_row = title_row + 2 
        start_col = title_col

        multiindex_offset = 1
        if isinstance(df.index, pd.MultiIndex):
            multiindex_offset = len(df.index.levels)
        
        # Write DataFrame row index
        for i, idx in enumerate(df.index):
            if isinstance(idx, tuple):
                for j, jdx in enumerate(idx):
                    try:
                        ws.cell(row=start_row + i, column=start_col+j, value=jdx)
                    except:
                        pass
            else:
                try:
                    ws.cell(row=start_row + i, column=start_col, value=idx)
                except:
                    pass

        # Write DataFrame columns and values
        for i, (idx, row) in enumerate(df.iterrows()):
            for j, value in enumerate(row):
                if pd.notnull(value):
                    ws.cell(row=start_row + i, column=start_col + j + multiindex_offset, value=value)

    # Save the updated Excel file
    wb.save(output_path)
    print(f"Data written successfully to {output_path}")

In [4]:
survey_args = {'dir':r'Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Processed_20241127\reformat_2019_rmoveonly_v2\04-merge_skims\adj_weights',
               'hh':    {'file':'survey2023_hrecx_rewt_base2023.dat',
                         'sep':' ',
                         'weight':'hhexpfac'},
               'person':{'file':'survey2023_precx_rewt_base2023.dat',
                         'sep':' ',
                         'weight':'psexpfac'},
               'day':   {'file':'survey2023_pdayx_rewt_base2023.dat',
                         'sep':' ',
                         'weight':'pdexpfac'},
               'trip':  {'file':'survey2023_tripx_rewt_base2023.dat',
                         'sep':' ',
                         'weight':'trexpfac'},
               'tour':  {'file':'survey2023_tourx_rewt_base2023.dat',
                         'sep':' ',
                         'weight':'toexpfac'},
#               'raw_dir': r'',
#               'raw_hh':     {'file':'hh.csv',
#                              'weight':'hh_weight'},
#               'raw_person': {'file':'person.csv',
#                              'weight':'person_weight'},
#               'raw_day':    {'file':'day.csv',
#                              'weight':'day_weight'},
#               'raw_trip':   {'file':'trip.csv',
#                              'weight':'trip_weight'},
             }

In [5]:
s23 = Survey(**SURVEY_2023)
ps23 = ProcessedSurvey(**survey_args)

In [6]:
flagged = flag_complete(s23, 
                    hh_unit='family',
                    concurrent_days=0,
                    concurrent_weighted_days=0)

Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\travel_diary_survey_scripts\completeness_evaluation\notebooks\survey_completeness.py:84: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  person.drop(columns=['sfcta_num_days_complete'], inplace=True)


In [8]:
hh = pd.merge(ps23.hh.data,
              flagged.hh.rename(columns={'hh_id':'hhno'})
              [['hhno',
                'num_concurrent_non_proxy_days',
                'sfcta_hh_complete',
                'sfcta_hh_concurrent_complete']])
person = pd.merge(ps23.person.data,
                  flagged.person.rename(columns={'hh_id':'hhno',
                                                 'person_num':'pno'})
                  [['hhno',
                    'pno',
                    'sfcta_num_days_complete', 
                    'sfcta_person_complete',]])
day = pd.merge(ps23.day.data,
               flagged.day.rename(columns={'hh_id':'hhno',
                                           'person_num':'pno',
                                           'travel_dow':'day'})
               [['hhno',
                 'pno',
                 'day',
                 'num_trips',
                 'sfcta_day_complete']])
day = pd.merge(day, person[['hhno','pno','pagey']], how='left')
day = pd.merge(day, hh[['hhno','hhtaz']], how='left')
day['no_travel'] = day['num_trips'].eq(0) * 1
day['is_typwkday'] = day['day'].isin([2,3,4]) * 1
day['is_adult'] = day['pagey'].ge(18) * 1


trip = pd.merge(ps23.trip.data,
                flagged.trip.rename(columns={'hh_id':'hhno',
                                             'person_num':'pno',
                                             'travel_dow':'day',
                                             'trip_num':'tsvid',
                                             })
                [['hhno','pno','day','tsvid',
                  'sfcta_is_complete']],
                how='left',
                on=['hhno','pno','day','tsvid'])
trip = pd.merge(trip, hh[['hhno','hhtaz']], how='left')
trip = pd.merge(trip, day[['hhno','pno','day','is_typwkday']], how='left', on=['hhno','pno','day'])

tour = pd.merge(ps23.tour.data.copy(),
                hh[['hhno','hhtaz']], how='left')
tour = pd.merge(tour, day[['hhno','pno','day','is_typwkday']])

In [9]:
df = pd.DataFrame(index=['Households',
                         '5-day Households',
                         'Persons',
                         'Days',
                         'Days (typical weekday)',
                         'Days (typical weekday, no travel)',
                         'Trips',
                         'Trips (typical weekday)',
                         'Tours',
                         'Tours (typical weekday)'],
                  columns=['SF','Non-SF'])

In [10]:
df.update(pd.DataFrame(hh.loc[hh['sfcta_hh_complete'].eq(1)].groupby(hh['hhtaz'].le(981)).size())
           .T
           .rename(columns={False:'Non-SF',True:'SF'},
                   index={0:'Households'}))

In [11]:
df.update(pd.DataFrame(hh.loc[hh['sfcta_hh_complete'].eq(1) & hh['num_concurrent_non_proxy_days'].ge(5)].groupby(hh['hhtaz'].le(981)).size())
           .T
           .rename(columns={False:'Non-SF',True:'SF'},
                   index={0:'5-day Households'}))

In [12]:
df.update(pd.DataFrame(person.loc[person['sfcta_person_complete'].eq(1)].groupby(person['hhtaz'].le(981)).size())
           .T
           .rename(columns={False:'Non-SF',True:'SF'},
                   index={0:'Persons'}))

In [13]:
df.update(pd.DataFrame(day.loc[day['sfcta_day_complete'].eq(1)].groupby(day['hhtaz'].le(981)).size())
           .T
           .rename(columns={False:'Non-SF',True:'SF'},
                   index={0:'Days'}))

In [14]:
df.update(pd.DataFrame(day.loc[day['sfcta_day_complete'].eq(1) & day['is_typwkday'].eq(1)].groupby(day['hhtaz'].le(981)).size())
           .T
           .rename(columns={False:'Non-SF',True:'SF'},
                   index={0:'Days (typical weekday)'}))

In [15]:
df.update(pd.DataFrame(day.loc[day['sfcta_day_complete'].eq(1) & day['no_travel'].eq(1) & day['is_typwkday'].eq(1)].groupby(day['hhtaz'].le(981)).size())
           .T
           .rename(columns={False:'Non-SF',True:'SF'},
                   index={0:'Days (typical weekday, no travel)'}))

In [16]:
df.update(pd.DataFrame(trip.loc[trip['sfcta_is_complete'].eq(1)].groupby(trip['hhtaz'].le(981)).size())
           .T
           .rename(columns={False:'Non-SF',True:'SF'},
                   index={0:'Trips'}))

In [17]:
df.update(pd.DataFrame(trip.loc[trip['sfcta_is_complete'].eq(1) & trip['is_typwkday'].eq(1)].groupby(trip['hhtaz'].le(981)).size())
           .T
           .rename(columns={False:'Non-SF',True:'SF'},
                   index={0:'Trips (typical weekday)'}))

In [18]:
df.update(pd.DataFrame(tour.groupby(tour['hhtaz'].le(981)).size())
           .T
           .rename(columns={False:'Non-SF',True:'SF'},
                   index={0:'Tours'}))

In [19]:
df.update(pd.DataFrame(tour.loc[tour['is_typwkday'].eq(1)].groupby(tour['hhtaz'].le(981)).size())
           .T
           .rename(columns={False:'Non-SF',True:'SF'},
                   index={0:'Tours (typical weekday)'}))

In [20]:
df['Total'] = df['SF'] + df['Non-SF']

In [21]:
table_data = {'BATS 2023 Sample Descriptive Statistics':df}
template_path = r'Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Documentation\Sample Descriptive Statistics (template).xlsx'
output_path = r'Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Documentation\Sample Descriptive Statistics.xlsx'

write_to_excel(template_path, output_path, table_data)

Data written successfully to Q:\Data\Surveys\HouseholdSurveys\MTC-SFCTA2022\Documentation\Sample Descriptive Statistics.xlsx
